# Titanic Survival Predictor

**What you'll learn:** how to choose features, clean real data, train two model types, and read an evaluation report.

**Estimated time:** 45–60 min &nbsp;|&nbsp; **Runtime:** CPU is fine

---

### Before you touch any code — think first

The Titanic dataset has these columns:
`survived, pclass, sex, age, sibsp, parch, fare, embarked, class, who, adult_male, deck, embark_town, alive, alone`

? **Your prediction:** Which 4–6 features do you think will best predict survival? Write them down before you see what we use. You'll compare at the end.


## 1  Setup & load data

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

df = sns.load_dataset('titanic')
print("Shape:", df.shape)
df.head()

### What are we looking at?

Each row is one passenger. The `survived` column is our **target** — the thing we want to predict. Every other column is a potential **feature** — an input our model can use.

? **Stop:** How many passengers are in this dataset? What fraction survived? Run the next cell to find out, then explain the numbers.


In [ ]:
print("Survival counts:")
print(df['survived'].value_counts())
print()
print("Missing values per column:")
print(df.isna().sum())

### Understanding the output

Two things to notice:
1. The dataset is **imbalanced** — more passengers died than survived. A dumb model that always predicts "died" would already be ~60% accurate. Our real goal is to beat that meaningfully.
2. `age` is missing for 177 passengers. We can't just ignore those rows — we need to handle the missing values.

? **Think:** Why is it problematic to just drop all rows where `age` is missing?


## 2  Choose and clean your features

Now you make the first real decision: **which columns to keep**.

We'll work with these: `pclass`, `sex`, `age`, `sibsp`, `parch`, `fare`.

Your job is to:
1. Select just those columns plus `survived`
2. Fill missing `age` values with the **median** age (not the mean — why? because age distributions are skewed)
3. Fill the one missing `fare` value with the median
4. Encode `sex` as a number: male -> 1, female -> 0

Fill in the `# YOUR CODE HERE` sections below.


In [ ]:
# Select the columns we want
# Hint: df[['col1', 'col2', ...]].copy()
cols = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']
data = df[cols].copy()

# Fill missing age with the median
# YOUR CODE HERE (one line: data['age'] = ...)
data['age'] = data['age'].fillna(data['age'].median())

# Fill missing fare with the median
# YOUR CODE HERE
data['fare'] = data['fare'].fillna(data['fare'].median())

# Encode sex: 'male' -> 1, 'female' -> 0
# Hint: (data['sex'] == 'male').astype(int)
# YOUR CODE HERE
data['sex'] = (data['sex'] == 'male').astype(int)

print("Missing values remaining:", data.isna().sum().sum())
data.head()

## 3  Engineer a new feature

Raw data rarely has all the signal you need. **Feature engineering** means creating new columns from existing ones that capture something the model can't see on its own.

Hypothesis: traveling alone might affect survival differently than traveling in a large group.

Two features to create:
- `family_size` = `sibsp` + `parch` + 1 (the passenger themselves)
- `is_alone` = 1 if family_size == 1, else 0

? **Predict:** Do you think being alone helped or hurt survival chances? Why?


In [ ]:
# Create family_size
# YOUR CODE HERE
data['family_size'] = data['sibsp'] + data['parch'] + 1

# Create is_alone (1 if alone, 0 otherwise)
# YOUR CODE HERE
data['is_alone'] = (data['family_size'] == 1).astype(int)

# Let's see the survival rate by is_alone to check your prediction
print("Survival rate when alone vs. not alone:")
print(data.groupby('is_alone')['survived'].mean().round(3))

### What did you find?

Compare the survival rates. Were you right in your prediction?

This is **exploratory data analysis** — you're building intuition before training anything. A good ML practitioner always does this step.


## 4  Split into train and test sets

We need to evaluate our model on data it has **never seen during training**. If we tested on the same data we trained on, we'd be grading our own exam with the answer key — meaningless.

The standard split: 80% train, 20% test. The `random_state` makes it reproducible.

Fill in the `test_size` and `random_state` parameters. Convention is `test_size=0.2, random_state=42`.


In [ ]:
X = data.drop(columns='survived')
y = data['survived']

# YOUR CODE HERE — fill in test_size and random_state
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")
print(f"Features:         {list(X.columns)}")

## 5  Train two models

We'll train two very different classifiers and compare them:

- **Logistic Regression** — a linear model. It draws a straight line (or hyperplane) through the feature space to separate survivors from non-survivors. Interpretable, fast, but limited.
- **Random Forest** — an ensemble of decision trees, each trained on a random data/feature subset. Handles non-linear patterns. Usually more powerful, but harder to interpret.

? **Before running:** Which model do you think will do better on this dataset, and why?


In [ ]:
# Train both models
logreg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
forest = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)

# Print test accuracy for both
for name, model in [('Logistic Regression', logreg), ('Random Forest', forest)]:
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{name}: {acc:.3f}")

### Reading the results

Accuracy alone doesn't tell the full story. Print the **classification report** for the better model.

The report shows:
- **Precision** — of all passengers we predicted "survived," what fraction actually did?
- **Recall** — of all passengers who actually survived, what fraction did we catch?
- **F1** — the harmonic mean of precision and recall; good when classes are imbalanced.

? **Think:** In a survival context, is it worse to miss a survivor (low recall) or falsely predict survival (low precision)?


In [ ]:
print("Classification Report — Random Forest:")
print(classification_report(y_test, forest.predict(X_test), target_names=['Died', 'Survived']))

## 6  What did your model actually learn?

In [ ]:
import pandas as pd
importances = pd.Series(forest.feature_importances_, index=X.columns)
importances.sort_values(ascending=False).plot.barh(figsize=(7, 4))
import matplotlib.pyplot as plt
plt.title("Feature importances (Random Forest)")
plt.tight_layout()
plt.show()
print(importances.sort_values(ascending=False).round(3))

? **Reflect:** Compare the feature importances to your prediction at the very top of the notebook. Were you right? Which feature surprised you most?

Notice that `sex` and `fare` dominate. This makes sense historically — women and first-class passengers were prioritized for lifeboats.


## 7  Your turn: cross-validation

A single train/test split is one random draw — we might have gotten lucky or unlucky. **Cross-validation** gives us a more honest estimate by training and testing on 5 different splits.

Fill in the `cross_val_score` call.


In [ ]:
# cross_val_score(estimator, X, y, cv=5, scoring='accuracy')
# YOUR CODE HERE — cross-validate the Random Forest on the full dataset
scores = cross_val_score(forest, X, y, cv=5, scoring='accuracy')
print(f"CV accuracy: {scores.mean():.3f} ± {scores.std():.3f}")
print(f"Per-fold:    {scores.round(3)}")

## 8  Challenge: improve the model

Try at least one of these and report whether it helped:

1. **Add more features:** create an `age_group` (child: age < 12, adult: else) — did children survive more?
2. **Tune the forest:** change `n_estimators` and `max_depth`. What's the tradeoff?
3. **Use only your original 4–6 features** (your gut-feel list from the top). How does accuracy compare?
4. **Find failure cases:** print the test rows where the forest was most wrong. Is there a pattern?

```python
# Starter for #4: find rows where the model was wrong
wrong_idx = X_test.index[forest.predict(X_test) != y_test]
print(data.loc[wrong_idx].head(10))
```

Good luck — and remember: the feature importances chart is the most important thing you built here.
